# VAE landscape — dense 40×40 grid (v2)

# VAE landscape of electrothermal battery charging
Three controlled experiments: (A) voltage trajectories at 25 °C; (B) surface-temperature trajectories at fixed 4.10 V; (C) joint concatenated voltage and temperature trajectories. Each has an independently trained two-dimensional VAE for direct visualization, and an optional higher-dimensional latent size. A second joint representation concatenates the two separately trained latent codes. Waveform libraries include deliberately poor protocols; no physical evaluation is used to train the VAE. Learn the waveform manifold, then label it with physical responses. Failed, incomplete and constraint-violating simulations are retained separately. No synthetic physics results are supplied.

In [ ]:
import sys, subprocess, os
print('VAE landscape v2 dense | Python:', sys.version)
if any(k in sys.modules for k in ('jax','jaxlib','pybamm')):
    raise RuntimeError('Use Runtime > Disconnect and delete runtime, then reconnect and run all.')
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pybamm==25.12.2'])
import importlib.util
if importlib.util.find_spec('jax') is not None:
    raise RuntimeError('JAX remains installed; use a clean dedicated runtime.')
os.environ['PYBAMM_DISABLE_TELEMETRY'] = 'true'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
import pybamm, time, json, pathlib
from scipy.stats import norm
from scipy.integrate import trapezoid
from IPython.display import display
assert pybamm.__version__ == '25.12.2', pybamm.__version__
pybamm.set_logging_level('ERROR')
torch.set_num_threads(1)
print('VAE landscape v2 dense ready:', pybamm.__version__, torch.__version__)


## 1. Configuration and common physics
Same DFN/OKane2022, partially reversible plating and lumped thermal model as the prior electrothermal experiments. Surface temperature is the imposed ambient boundary (not resolved surface physics). All protocols start at 20% SOC and 25 °C and stop at the 80% stored-SOC proxy or the horizon. Command bounds intentionally include low voltage, cold and hot conditions; these are numerical experiments, not certified safe protocols. Default 96 physical evaluations per experiment, plus 1,600 decoded grid evaluations per experiment and smoke tests. Set GRID_SIDE=0 to disable the grid, or increase for publication. Dense grid results are checkpointed after every evaluated point. Optionally use a persistent Google Drive output directory before starting. Training library is independent of expensive evaluations.

In [ ]:
HORIZON=10800.
RAMP=1.
V_MIN,V_MAX=3.4,4.2
I_MAX=15.
SOC0,SOC_TARGET=.2,.8
TEMP=298.15
MODEL='DFN'
BASE_SEED=42
MIN_DT=60.
MAX_SEG=6
THERM_BLOCKS=12
T_MIN,T_MAX=288.15,318.15
CELL_T_MAX=323.15
T_NORM=(TEMP-T_MIN)/(T_MAX-T_MIN)
N_WAVE=64
N_LIBRARY=4800
N_EVAL=96
LATENT=2
EPOCHS=100
BATCH=128
BETA=.005
GRID_SIDE=40
ATLAS_SIDE=8
GRID_PLOT_EVERY=100
GRID_SAVE_TRACES=False  # all scalar outputs saved; optional per-point compressed raw traces
GRID_RESUME=True
PERSIST_TO_DRIVE=False  # set True BEFORE first run to preserve checkpoints across Colab sessions
CV_FIXED=4.10
OBJECTIVE='gross_mAh'
# Fixed hypervolume box: preliminary scales, freeze before comparison.
REF=np.array([180.,100.]) # minutes, mAh (for gross/peak/dead)
EXPOSURE_REF=18000. # mAh min
ROTATE_RISK=True # alternate risk metric every generation; set False for gross-only comparison
BASE_SEED=42
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=pathlib.Path('/content/drive/MyDrive/vae_electrothermal_dense_v2')
else:
    OUT=pathlib.Path('vae_electrothermal_dense_v2')
OUT.mkdir(parents=True,exist_ok=True)
CONC = 'Negative electrode volume-averaged concentration [mol.m-3]'
PLATE = 'Loss of capacity to negative lithium plating [A.h]'

def make_model(kind=MODEL):
    return getattr(pybamm.lithium_ion, kind)(options={'lithium plating':'partially reversible','thermal':'lumped','surface temperature':'ambient'})
def params():
    p = pybamm.ParameterValues('OKane2022')
    p.update({'Ambient temperature [K]':TEMP,'Initial temperature [K]':TEMP,
              'Upper voltage cut-off [V]':4.21})
    return p

def solver(): return pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-8)
def series(sol, name):
    a = np.asarray(sol[name].entries).squeeze()
    if a.ndim != 1 or len(a) != len(sol.t): raise ValueError(f'Unexpected shape {name}: {a.shape}')
    return a

def calibrate(soc):
    m = make_model(); p=params()
    exp=pybamm.Experiment([pybamm.step.current(0.,duration=1.,period=1.)])
    sol=pybamm.Simulation(m,parameter_values=p,experiment=exp,solver=solver()).solve(initial_soc=soc)
    return float(series(sol,CONC)[0]),float(series(sol,'Voltage [V]')[0])
C0, OCV0 = calibrate(SOC0)
CT, OCVT = calibrate(SOC_TARGET)
assert CT > C0
print('Calibration:', C0, CT, OCV0, OCVT)
from copy import deepcopy
from scipy.integrate import cumulative_trapezoid
from IPython.display import clear_output
RISK={'gross_mAh':'Cumulative net deposition (mAh)','peak_mAh':'Peak metallic lithium (mAh)','dead_mAh':'Dead lithium (mAh)','exposure_mAh_min':'Integrated plating exposure (mAh min)'}
assert OBJECTIVE in RISK
assert N_EVAL>=12 and N_LIBRARY>=N_EVAL and LATENT>=2

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
rng=np.random.default_rng(BASE_SEED);torch.manual_seed(BASE_SEED)
XTIME=np.linspace(0,1,N_WAVE)


## 2. Broad waveform libraries
Twelve balanced families: constants, rising/falling ramps, smooth multistage ramps, abrupt steps, Gaussian peaks, Gaussian dips, pulse trains, sine waves, short/long correlation GP samples, early high/late low and early low/late high profiles. Independent draws for joint library plus explicit stress corners (cold/high voltage, hot/high voltage, low voltage/cold, high-voltage pulses with cold periods). Values stay inside command bounds; physical infeasibility is measured rather than removed. A wide voltage range includes discharge and incomplete charging; high voltage and cold temperature may produce plating or excessive current. The libraries are generated without simulation.

In [ ]:
KINDS=['constant','ramp_up','ramp_down','multistage','step','gaussian_peak','gaussian_dip','pulse','sine','gp','early_high','late_high']
def wave(kind,rng):
    x=XTIME;a,b=rng.uniform(.02,.98,2);lo,hi=sorted([a,b])
    if kind=='constant':y=np.full(N_WAVE,a)
    elif kind=='ramp_up':y=lo+(hi-lo)*x
    elif kind=='ramp_down':y=hi-(hi-lo)*x
    elif kind=='multistage':y=np.interp(x,np.r_[0,np.sort(rng.uniform(.05,.95,4)),1],rng.uniform(0,1,6))
    elif kind=='step':y=np.where(x<rng.uniform(.1,.9),a,b)
    elif kind in ('gaussian_peak','gaussian_dip'):y=a+(1 if kind=='gaussian_peak' else -1)*rng.uniform(.25,.9)*np.exp(-.5*((x-rng.uniform(.1,.9))/rng.uniform(.03,.22))**2)
    elif kind=='pulse':y=np.where((x*rng.integers(2,7))%1<rng.uniform(.15,.8),hi,lo)
    elif kind=='sine':y=.5+rng.uniform(.15,.5)*np.sin(2*np.pi*rng.integers(1,5)*x+rng.uniform(0,2*np.pi))
    elif kind=='gp':
        ell=rng.uniform(.035,.5);K=.28**2*np.exp(-.5*((x[:,None]-x[None,:])/ell)**2)+1e-8*np.eye(N_WAVE);y=a+np.linalg.cholesky(K)@rng.normal(size=N_WAVE)
    elif kind=='early_high':y=np.where(x<rng.uniform(.15,.55),rng.uniform(.8,1),rng.uniform(.02,.4))
    elif kind=='late_high':y=np.where(x<rng.uniform(.15,.55),rng.uniform(.02,.4),rng.uniform(.8,1))
    return np.clip(y,0,1).astype('float32')
def library(seed,channel):
    r=np.random.default_rng(seed);w=[];labels=[]
    for i in range(N_LIBRARY):
        kind=KINDS[i%len(KINDS)];w.append(wave(kind,r));labels.append(kind)
    return np.stack(w),np.array(labels)
V,VL=library(BASE_SEED,'V');T,TL=library(BASE_SEED+1,'T')
rr=np.random.default_rng(BASE_SEED+2);perm=rr.permutation(N_LIBRARY)
JV,JT=V.copy(),T[perm].copy();JL=np.char.add(np.char.add(VL,' / '),TL[perm])
# Independent, deliberately difficult joint corners, with provenance labels.
for i in range(N_LIBRARY//8):
    j=N_LIBRARY-1-i; x=XTIME; k=i%4
    if k==0:JV[j]=rr.uniform(.85,1);JT[j]=rr.uniform(0,.15);lab='cold/high V'
    elif k==1:JV[j]=rr.uniform(.85,1);JT[j]=rr.uniform(.85,1);lab='hot/high V'
    elif k==2:JV[j]=rr.uniform(0,.35);JT[j]=rr.uniform(0,.2);lab='cold/low V'
    else:JV[j]=np.where((x*4)%1<.5,1,.25);JT[j]=np.where((x*4)%1<.5,.05,.7);lab='cold/high pulse'
    JL[j]=lab
LIB={'voltage':V,'thermal':T,'joint':np.concatenate([JV,JT],axis=1)}
LABEL={'voltage':VL,'thermal':TL,'joint':JL}
# Hold all shared source rows out of both standalone and joint training when validating joint product codes.
# Independent within-experiment train/test split for VAE reconstruction (not physical performance prediction).
fig,ax=plt.subplots(2,6,figsize=(16,6))
for p,kind in zip(ax.flat,KINDS):
    ids=np.flatnonzero(VL==kind)[:12]
    for i in ids:p.plot(XTIME,V_MIN+(V_MAX-V_MIN)*V[i],alpha=.55)
    p.set(title=kind,ylabel='V',xlabel='Normalized time')
fig.tight_layout();fig.savefig(OUT/'voltage_library.png',dpi=160);plt.show()
fig,ax=plt.subplots(2,6,figsize=(16,6))
for p,kind in zip(ax.flat,KINDS):
    for i in np.flatnonzero(TL==kind)[:12]:p.plot(XTIME,T_MIN-273.15+(T_MAX-T_MIN)*T[i],alpha=.55)
    p.set(title=kind,ylabel='Surface °C',xlabel='Normalized time')
fig.tight_layout();fig.savefig(OUT/'thermal_library.png',dpi=160);plt.show()
np.savez_compressed(OUT/'libraries.npz',V=V,T=T,JV=JV,JT=JT,VL=VL,TL=TL,JL=JL)


## 3. Train independent and joint VAEs
Normalized amplitudes in [0,1] give equal scale to both channels. Encoder returns Gaussian posterior mean and log variance. Decoder sigmoid constrains command bounds. Loss is mean squared reconstruction plus beta KL (KL summed over latent dimensions). Train/test splits fixed by seed. Joint VAE is trained on concatenated waveform arrays, not on concatenated latent codes; both representations are analyzed separately. A two-dimensional latent space allows direct maps; for larger LATENT dimensions use PCA for display only. Physical labels are not included in VAE training.

In [ ]:
class VAE(nn.Module):
    def __init__(self,d):
        super().__init__();self.enc=nn.Sequential(nn.Linear(d,128),nn.SiLU(),nn.Linear(128,64),nn.SiLU());self.mu=nn.Linear(64,LATENT);self.lv=nn.Linear(64,LATENT);self.dec=nn.Sequential(nn.Linear(LATENT,64),nn.SiLU(),nn.Linear(64,128),nn.SiLU(),nn.Linear(128,d),nn.Sigmoid())
    def encode(self,x):
        h=self.enc(x);return self.mu(h),self.lv(h).clamp(-8,6)
    def forward(self,x):
        mu,lv=self.encode(x);z=mu+torch.randn_like(mu)*torch.exp(.5*lv);return self.dec(z),mu,lv
def fit_vae(x,seed):
    torch.manual_seed(seed);model=VAE(x.shape[1]);opt=torch.optim.Adam(model.parameters(),lr=.001)
    tr,te=train_test_split(np.arange(len(x)),test_size=.2,random_state=seed)
    xx=torch.tensor(x,dtype=torch.float32);loader=DataLoader(TensorDataset(xx[tr]),batch_size=BATCH,shuffle=True);history=[]
    for ep in range(EPOCHS):
        model.train();losses=[]
        for (a,) in loader:
            b,mu,lv=model(a);rec=(a-b).square().mean();kl=-.5*(1+lv-mu.square()-lv.exp()).sum(1).mean();loss=rec+BETA*kl
            opt.zero_grad();loss.backward();opt.step();losses.append([rec.item(),kl.item()])
        model.eval()
        with torch.no_grad():
            mu,lv=model.encode(xx[te]);test=(model.dec(mu)-xx[te]).square().mean().item()
        history.append([ep+1,*np.mean(losses,axis=0),test])
    with torch.no_grad():z=model.encode(xx)[0].numpy();rec=model.dec(torch.tensor(z)).numpy()
    return model,z,rec,np.array(history),tr,te
models={};Z={};RECON={};SPLIT={};proj={};Z2={}
for k,x in LIB.items():
    model,z,rec,h,tr,te=fit_vae(x,BASE_SEED)
    models[k]=model;Z[k]=z;RECON[k]=rec;SPLIT[k]=(tr,te)
    proj[k]=PCA(2).fit(z[tr]) if LATENT>2 else None;Z2[k]=proj[k].transform(z) if proj[k] else z
    torch.save(model.state_dict(),OUT/f'{k}_vae.pt')
    np.savez_compressed(OUT/f'{k}_latent.npz',z=z,reconstruction=rec,train=tr,test=te)
    pd.DataFrame(h,columns=['epoch','train_mse','train_kl','test_mse']).to_csv(OUT/f'{k}_training.csv',index=False)
    fig,ax=plt.subplots(1,3,figsize=(14,4));ax[0].plot(h[:,0],h[:,1],label='Train');ax[0].plot(h[:,0],h[:,3],label='Test');ax[0].legend();ax[0].set(title=k+' reconstruction loss',xlabel='Epoch')
    ax[1].plot(h[:,0],h[:,2]);ax[1].set(title='KL',xlabel='Epoch')
    mse=((rec-x)**2).mean(1);ax[2].scatter(*Z2[k][te].T,c=mse[te],s=6);ax[2].set(title='Held-out reconstruction error')
    fig.tight_layout();fig.savefig(OUT/f'{k}_training.png',dpi=160);plt.show()
    print(k,'held-out RMSE',np.sqrt(mse[te].mean()),'latent std',z[te].std(0))
    fig,ax=plt.subplots(2,6,figsize=(16,5))
    for p,i in zip(ax.flat,te[:12]):
        p.plot(x[i],label='original');p.plot(rec[i],'--',label='decoded');p.set_title(str(LABEL[k][i])[:30],fontsize=8)
    ax.flat[0].legend(fontsize=7);fig.tight_layout();fig.savefig(OUT/f'{k}_recon.png',dpi=160);plt.show()
def encode_mean(model,x):
    with torch.no_grad():return model.encode(torch.tensor(x,dtype=torch.float32))[0].numpy()
Z_product=np.concatenate([encode_mean(models['voltage'],JV),encode_mean(models['thermal'],JT)],axis=1)
product_pca=PCA(2).fit(Z_product);PRODUCT2=product_pca.transform(Z_product)
np.savez_compressed(OUT/'product_latent.npz',z=Z_product,z2=PRODUCT2)


## 4. Physical simulation and risk extraction
The evaluator uses the same electrothermal DFN as previous notebooks, with 12 piecewise-held surface-temperature subintervals. Within each subinterval the voltage waveform retains its original clock. Constant temperature trajectories use one step to reduce cost. Separate surface boundary and internal temperature are checked. Solver errors and early physical stops are not assigned fake objective values. Metrics for valid but incomplete trajectories are retained as observed-to-stop values and are never interpreted as completed charge time. Four plating metrics, peak current, progress, temperature, and mass-balance residual are recorded.

In [ ]:
def decode(s,t):return np.clip(np.interp(t,XTIME,np.asarray(s['a'])),0,1)
def protocol(x,kind):
    if kind=='voltage':v=x;temp=np.full(N_WAVE,T_NORM)
    elif kind=='thermal':v=np.full(N_WAVE,(CV_FIXED-V_MIN)/(V_MAX-V_MIN));temp=x
    else:v,temp=np.split(x,2)
    return [dict(mode='V',d=HORIZON,family='direct',a=np.asarray(v).tolist(),temp=np.asarray(temp).tolist())]
m=make_model(); p=params()
def resolve(*names):
    for name in names:
        if name in m.variables: return name
    raise KeyError(f'Missing {names}; available: '+str([k for k in m.variables if 'plating' in k.lower() or 'dead lithium' in k.lower()]))
MET=resolve('Negative lithium plating concentration [mol.m-3]','Negative electrode lithium plating concentration [mol.m-3]','Lithium plating concentration [mol.m-3]')
DEAD=resolve('Negative dead lithium concentration [mol.m-3]','Negative electrode dead lithium concentration [mol.m-3]','Dead lithium concentration [mol.m-3]')
FLUX=resolve('Negative electrode lithium plating volumetric interfacial current density [A.m-3]','Lithium plating volumetric interfacial current density [A.m-3]')
AREA=float(p.evaluate(m.param.L_y*m.param.L_z)); F=float(p.evaluate(m.param.F))
print('Resolved:',MET,DEAD,FLUX, 'Area:',AREA)
def drive_segment(s):
    d=s['d']
    if s['family']=='constant':
        a=np.asarray(s['a']);t=[0.];y=[a[0]]
        for k in range(1,len(a)):
            edge=d*k/len(a);t.extend([edge-min(RAMP,d/len(a)/2),edge]);y.extend([a[k-1],a[k]])
        t.append(d);y.append(a[-1]);t=np.asarray(t);y=np.clip(y,0,1)
    else:
        t=np.linspace(0,d, N_WAVE);y=decode(s,t/d)
    u=-I_MAX*y if s['mode']=='I' else V_MIN+(V_MAX-V_MIN)*y
    return np.column_stack([t,u])
def thermal_steps(protocol):
    # Native PyBaMM surface temperature = ambient boundary temperature.
    # Electrical waveform retains its local clock when sliced; only thermal control is zero-order held.
    for s in protocol:
        drive=drive_segment(s); edges=np.linspace(0,s['d'],2 if np.ptp(s['temp'])<1e-8 else THERM_BLOCKS+1)
        for lo,hi in zip(edges[:-1],edges[1:]):
            inner=drive[(drive[:,0]>lo)&(drive[:,0]<hi),0]
            t=np.r_[lo,inner,hi]; u=np.interp(t,drive[:,0],drive[:,1])
            temp=decode(dict(s,a=s['temp']),np.array([(lo+hi)/(2*s['d'])]))[0]
            yield dict(s,d=float(hi-lo),drive=np.column_stack([t-lo,u]),surface_K=float(T_MIN+(T_MAX-T_MIN)*temp))
def termination():
    return pybamm.step.CustomTermination('Stored charge target',lambda v:CT-v[CONC])
def spatial(sol,name,dx):
    a=np.asarray(sol[name].entries)
    if a.shape!=(len(dx),len(sol.t)): raise ValueError(f'{name} shape {a.shape}, expected {(len(dx),len(sol.t))}')
    return a
def evaluate(protocol,trace=False):
    start=time.perf_counter(); chunks=[]; previous=None; wall=0.; reached=False;error=''
    try:
        for s in thermal_steps(protocol):
            constructor=pybamm.step.current if s['mode']=='I' else pybamm.step.voltage
            st=constructor(s['drive'],duration=s['d'],period=2.,termination=termination(),skip_ok=False)
            par=params();par.update({'Ambient temperature [K]':s['surface_K']})
            sim=pybamm.Simulation(make_model(),parameter_values=par,solver=solver(),experiment=pybamm.Experiment([st]))
            kwargs={'initial_soc':SOC0} if previous is None else {'starting_solution':previous}
            previous=sim.solve(calc_esoh=False,**kwargs)
            sol=previous.cycles[-1].steps[-1]
            t=np.asarray(sol.t); C=series(sol,CONC); I=series(sol,'Current [A]');V=series(sol,'Voltage [V]')
            T=series(sol,'Volume-averaged cell temperature [K]')
            TS=series(sol,'Surface temperature [K]')
            if not np.allclose(TS,s['surface_K'],atol=1e-4):raise ValueError('Surface boundary mismatch')
            if not chunks and abs(T[0]-TEMP)>1e-3:raise ValueError('Initial cell temperature mismatch')
            if chunks and abs(T[0]-chunks[-1]['T'][-1])>1e-3:raise ValueError('Thermal state continuity mismatch')
            dx=np.diff(sim.mesh['negative electrode'].edges)
            metal=spatial(sol,MET,dx); dead=spatial(sol,DEAD,dx); j=spatial(sol,FLUX,dx)
            Q=AREA*F/3.6*(dx@metal); D=AREA*F/3.6*(dx@dead)
            G=AREA/3.6*(dx@np.maximum(-j,0)); net=AREA/3.6*(dx@(-j))
            if not np.isfinite(np.stack([t,C,I,V,Q,D,G,net,T,TS])).all(): raise ValueError('Nonfinite trace')
            hit=np.flatnonzero(C>=CT-1e-5*(CT-C0)); k=int(hit[0])+1 if len(hit) else len(t)
            chunks.append(dict(t=t[:k],C=C[:k],I=I[:k],V=V[:k],Q=Q[:k],D=D[:k],G=G[:k],net=net[:k],mode=np.full(k,s['mode']),T=T[:k],TS=TS[:k],local=np.max(metal[:,:k],axis=0)))
            reached=bool(len(hit))
            if reached: break
            if t[-1]-t[0]<s['d']-0.05: raise ValueError('Unscheduled early step termination: '+str(sol.termination))
        a={key:np.concatenate([c[key] for c in chunks]) for key in chunks[0]}
        t,C,I,V,Q,D=[a[key] for key in ('t','C','I','V','Q','D')]
        # Integrate separately per segment, avoiding quadrature across command discontinuities.
        gross=sum(trapezoid(c['G'],c['t']) for c in chunks)
        net=sum(trapezoid(c['net'],c['t']) for c in chunks)
        exposure=sum(trapezoid(np.maximum(c['Q'],0),c['t'])/60 for c in chunks)
        residual=float(net-(Q[-1]+D[-1]-Q[0]-D[0]))
        progress=float((C[-1]-C0)/(CT-C0)); imax=float(np.max(np.abs(I)))
        violation=max(0,(a['T'].max()-CELL_T_MAX)/10)+max(0,1-progress)+max(0,imax/I_MAX-1)+max(0,(np.max(V)-V_MAX)/.1)+max(0,(V_MIN-np.min(V))/.1)
        feasible=bool(reached and a['T'].max()<=CELL_T_MAX+1e-3 and imax<=I_MAX+1e-3 and np.max(V)<=V_MAX+1e-5 and np.min(V)>=V_MIN-1e-5)
        row=dict(feasible=feasible,violation=violation,time_min=float(t[-1]/60),gross_mAh=float(gross),peak_mAh=float(max(0,Q.max())),dead_mAh=float(max(0,D[-1]-D[0])),exposure_mAh_min=float(exposure),final_mAh=float(Q[-1]),balance_residual_mAh=residual,max_A=imax,progress=progress,local_peak=float(a['local'].max()),voltage_fraction=sum(c['t'][-1]-c['t'][0] for c in chunks if c['mode'][0]=='V')/max(t[-1],1e-9),segments_used=len(chunks),max_cell_K=float(a['T'].max()),surface_exposure_K_min=float(sum(trapezoid(np.abs(c['TS']-TEMP),c['t'])/60 for c in chunks)),error='',wall_s=time.perf_counter()-start)
        if trace: row['trace']=a
        return row
    except Exception as e:
        return dict(feasible=False,violation=100.,time_min=np.nan,**{k:np.nan for k in RISK},error=f'{type(e).__name__}: {e}',wall_s=time.perf_counter()-start)

for kind,x in [('voltage',np.full(N_WAVE,(CV_FIXED-V_MIN)/(V_MAX-V_MIN))),('thermal',np.linspace(T_NORM,.7,N_WAVE))]:
    result=evaluate(protocol(x,kind),trace=True);print(kind,{k:v for k,v in result.items() if k!='trace'})
    if result['error']:raise RuntimeError(result['error'])


## 5. Evaluate a diverse subset of the waveform library
Select points from held-out data, with stratification across waveform families, explicit extreme protocols, and greedy latent-space coverage. Selection is independent of physical reward. This yields a map of properties, not an optimized Pareto front. Infeasible protocols and failed solves are part of the dataset. Evaluate the original waveform, not the decoded approximation. In the next section evaluate decoded trajectories separately to measure what the VAE can actually generate.

In [ ]:
def select_ids(k,n):
    x,z=LIB[k],Z2[k];test=SPLIT[k][1];labels=LABEL[k];r=np.random.default_rng(BASE_SEED)
    chosen=[]
    groups=KINDS if k!='joint' else ['cold/high V','hot/high V','cold/low V','cold/high pulse']
    for lab in groups:
        ids=test[labels[test]==lab]
        if len(ids):chosen.extend(r.choice(ids,min(3,len(ids)),replace=False).tolist())
    # Extremes in each input channel.
    for j in range(0,x.shape[1],N_WAVE):
        y=x[test,j:j+N_WAVE];chosen.extend([int(test[y.mean(1).argmin()]),int(test[y.mean(1).argmax()]),int(test[np.ptp(y,axis=1).argmax()])])
    chosen=list(dict.fromkeys(chosen))[:n]
    zz=(z-z[test].mean(0))/np.maximum(z[test].std(0),1e-6)
    d=np.min(((zz[test,None,:]-zz[np.array(chosen)][None,:,:])**2).sum(2),axis=1)
    while len(chosen)<min(n,len(test)):
        d[np.isin(test,chosen)]=-1;i=int(np.argmax(d));j=int(test[i]);chosen.append(j);d=np.minimum(d,((zz[test]-zz[j])**2).sum(1))
    return chosen
def classify(r):
    if r['error']:return 'solver/event failure'
    if r['feasible']:return 'feasible'
    if r.get('max_A',0)>I_MAX+1e-3:return 'current limit'
    if r.get('max_cell_K',0)>CELL_T_MAX+1e-3:return 'temperature limit'
    if r.get('progress',0)<1-1e-5:return 'incomplete SOC'
    return 'other constraint'
def evaluate_batch(k,items,tag):
    rows=[];traces={}
    for j,(idx,x,z) in enumerate(items):
        rr=evaluate(protocol(x,k),trace=True);tr=rr.pop('trace',None)
        if tr is not None:traces.update({f'{j}_{name}':val for name,val in tr.items()})
        row=dict(index=idx,source=tag,z1=float(z[0]),z2=float(z[1]),family=str(LABEL[k][idx]) if idx>=0 else 'decoded',status=classify(rr),**rr)
        rows.append(row)
        if (j+1)%8==0 or j==len(items)-1:
            pd.DataFrame(rows).to_csv(OUT/f'{k}_{tag}_evaluations.csv',index=False);np.savez_compressed(OUT/f'{k}_{tag}_traces.npz',**traces)
            print(k,tag,j+1,'/',len(items),pd.Series([r['status'] for r in rows]).value_counts().to_dict(),flush=True)
    return pd.DataFrame(rows)
observed={}
for k in LIB:
    ids=select_ids(k,N_EVAL);observed[k]=evaluate_batch(k,[(i,LIB[k][i],Z2[k][i]) for i in ids],'original')
    display(observed[k].status.value_counts())


## 6. Physical property maps of the latent spaces
Plot real simulation labels on top of the complete latent distribution. Separate feasibility, failure, progress, peak current, time to target, four plating metrics and peak temperature. Continuous maps are scatter plots, not fabricated interpolation. Completed charging metrics are only displayed for feasible cases; incomplete cases have their own progress map. The joint VAE and concatenated independent codes are compared using the same physical evaluations.

In [ ]:
FIELDS={'time_min':'Time to target (min)',**RISK,'progress':'Target progress','max_A':'Peak |I| (A)','max_cell_K':'Peak cell temperature (K)'}
def property_map(k,df,background,tag):
    fig,ax=plt.subplots(3,3,figsize=(15,12));p=ax.flat[0]
    p.scatter(*background.T,s=2,c='lightgray',alpha=.2)
    for name,g in df.groupby('status'):p.scatter(g.z1,g.z2,s=25,label=name)
    p.set_title('Feasibility / failure');p.legend(fontsize=7)
    for p,(field,label) in zip(ax.flat[1:],FIELDS.items()):
        p.scatter(*background.T,s=2,c='lightgray',alpha=.15)
        if field in df:
            valid=df[field].notna()
            if field=='time_min' or field in RISK:valid &= df.feasible
            g=df[valid]
            if len(g):
                v=g[field].to_numpy();color=np.log1p(np.maximum(v,0)) if field in RISK else v
                q=p.scatter(g.z1,g.z2,c=color,s=26,cmap='viridis');fig.colorbar(q,ax=p,label='log(1+value)' if field in RISK else label)
        p.set_title(label)
    for p in ax.flat:p.set(xlabel='Latent coordinate 1',ylabel='Latent coordinate 2')
    fig.suptitle(k+' | '+tag);fig.tight_layout();fig.savefig(OUT/f'{k}_{tag}_maps.png',dpi=170);plt.show()
for k,df in observed.items():property_map(k,df,Z2[k],'original')
df=observed['joint'].copy();ids=df['index'].to_numpy();df[['z1','z2']]=PRODUCT2[ids];property_map('joint_product',df,PRODUCT2,'original')


## 7. Dense decoded latent-space grid (40 × 40 per VAE)
Every grid point is decoded and physically evaluated, not interpolated. Default: 1,600 points per VAE, 4,800 total. The grid spans the central 90% marginal latent coordinate ranges, including possible off-manifold corners. Nearest-neighbor distance is saved. For higher latent dimensions this is a PCA slice. Display a representative 8 × 8 waveform atlas only, and dense heatmaps with unsimulated/failed cells masked. Update plots every 100 evaluations; checkpoint scalar results after every point. On rerun, resume only if the decoded waveform grid and model configuration match the saved fingerprint; otherwise stop to prevent mixing results. For persistent checkpoints across Colab resets set PERSIST_TO_DRIVE=True before execution. Raw traces are optional and disabled for dense grid to control storage.

In [ ]:
import hashlib, csv
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
STATUS=['feasible','current limit','temperature limit','incomplete SOC','other constraint','solver/event failure']
STATUS_COLORS=['#009E73','#E69F00','#CC79A7','#56B4E9','#D55E00','#666666']
def dense_maps(k,df,gx,gy):
    fig,ax=plt.subplots(3,3,figsize=(15,12))
    extent=[gx[0],gx[-1],gy[0],gy[-1]]
    for p,(field,label) in zip(ax.flat,[('status','Feasibility / failure')]+list(FIELDS.items())):
        a=np.full((len(gy),len(gx)),np.nan)
        for _,row in df.iterrows():
            j,i=int(row.grid_y),int(row.grid_x)
            if field=='status':
                if row.status in STATUS:a[j,i]=STATUS.index(row.status)
            elif field in row and pd.notna(row[field]):
                if field=='time_min' or field in RISK:
                    if not bool(row.feasible):continue
                val=float(row[field]);a[j,i]=np.log1p(max(0,val)) if field in RISK else val
        if field=='status':
            cm=ListedColormap(STATUS_COLORS);cm.set_bad('#eeeeee')
            p.imshow(np.ma.masked_invalid(a),origin='lower',extent=extent,aspect='auto',interpolation='nearest',cmap=cm,norm=BoundaryNorm(np.arange(-.5,len(STATUS)+.5),len(STATUS)))
            p.legend(handles=[Patch(color=c,label=s) for s,c in zip(STATUS,STATUS_COLORS)]+[Patch(color='#eeeeee',label='Not evaluated')],fontsize=6,loc='best')
        else:
            cm=plt.get_cmap('viridis').copy();cm.set_bad('#eeeeee')
            im=p.imshow(np.ma.masked_invalid(a),origin='lower',extent=extent,aspect='auto',interpolation='nearest',cmap=cm)
            if np.isfinite(a).any():fig.colorbar(im,ax=p,label='log(1+value)' if field in RISK else label)
        p.set(title=label,xlabel='Latent coordinate 1',ylabel='Latent coordinate 2')
    fig.suptitle(f'{k}: {len(df)}/{len(gx)*len(gy)} physical grid evaluations')
    fig.tight_layout();fig.savefig(OUT/f'{k}_dense_maps.png',dpi=170);plt.show();plt.close(fig)
def atomic_csv(df,path):
    tmp=path.with_suffix('.tmp');df.to_csv(tmp,index=False);os.replace(tmp,path)
def run_dense(k,w,g2,g,d,gx,gy):
    path=OUT/f'{k}_decoded_evaluations.csv';manifest=OUT/f'{k}_grid_manifest.json'
    settings=dict(version='dense-v2',pybamm=pybamm.__version__,model=MODEL,parameter_set='OKane2022',soc0=SOC0,target=SOC_TARGET,initial_K=TEMP,horizon=HORIZON,voltage_bounds=[V_MIN,V_MAX],temperature_bounds=[T_MIN,T_MAX],current_limit=I_MAX,cell_temperature_limit=CELL_T_MAX,thermal_blocks=THERM_BLOCKS,cv=CV_FIXED,grid_side=GRID_SIDE)
    digest=hashlib.sha256()
    for a in (w,g2,g):digest.update(np.ascontiguousarray(a).tobytes())
    settings['waveform_grid_sha256']=digest.hexdigest()
    if path.exists() and GRID_RESUME:
        if not manifest.exists() or json.loads(manifest.read_text())!=settings:
            raise RuntimeError('Checkpoint grid/settings mismatch; choose a new OUT directory or disable GRID_RESUME to restart.')
        rows=pd.read_csv(path,keep_default_na=False,na_values=['']).to_dict('records')
        for row in rows:
            row['feasible']=str(row['feasible']).lower()=='true'
            if pd.isna(row.get('error')):row['error']=''
        done={int(row['grid_id']) for row in rows}
        print(k,'resuming',len(done),'completed grid points',flush=True)
    else:
        rows=[];done=set()
        if path.exists():path.unlink()
        manifest.write_text(json.dumps(settings,indent=2))
    trace_dir=OUT/f'{k}_grid_traces'
    if GRID_SAVE_TRACES:trace_dir.mkdir(exist_ok=True)
    for j,(x,z) in enumerate(zip(w,g2)):
        if j in done:continue
        rr=evaluate(protocol(x,k),trace=GRID_SAVE_TRACES);trace=rr.pop('trace',None)
        row=dict(index=-1,source='decoded',grid_id=j,grid_x=j%GRID_SIDE,grid_y=j//GRID_SIDE,z1=float(z[0]),z2=float(z[1]),family='decoded',status=classify(rr),latent_nn_distance=float(d[j]),**rr)
        if trace is not None:np.savez_compressed(trace_dir/f'{j:05d}.npz',**trace)
        rows.append(row);done.add(j)
        frame=pd.DataFrame(rows).sort_values('grid_id');atomic_csv(frame,path)
        if len(done)%GRID_PLOT_EVERY==0 or len(done)==len(w):
            print(k,len(done),'/',len(w),frame.status.value_counts().to_dict(),flush=True)
            dense_maps(k,frame,gx,gy)
    frame=pd.DataFrame(rows).sort_values('grid_id')
    dense_maps(k,frame,gx,gy)
    return frame
decoded={}
for k in LIB:
    if GRID_SIDE<=0:continue
    tr=SPLIT[k][0];lo,hi=np.quantile(Z2[k][tr],[.05,.95],axis=0)
    gx=np.linspace(lo[0],hi[0],GRID_SIDE);gy=np.linspace(lo[1],hi[1],GRID_SIDE)
    g2=np.array([(a,b) for b in gy for a in gx]);g=proj[k].inverse_transform(g2) if proj[k] else g2
    with torch.no_grad():w=models[k].dec(torch.tensor(g,dtype=torch.float32)).numpy()
    d=NearestNeighbors(n_neighbors=1).fit(Z[k][tr]).kneighbors(g)[0].ravel()
    # Save complete decoded grid only after ensuring any existing results refer to exactly this grid.
    df=run_dense(k,w,g2,g,d,gx,gy);decoded[k]=df
    np.savez_compressed(OUT/f'{k}_decoded_waveforms.npz',coordinates=g2,latent=g,waveforms=w,distance=d,gx=gx,gy=gy)
    ix=np.unique(np.linspace(0,GRID_SIDE-1,min(ATLAS_SIDE,GRID_SIDE)).round().astype(int))
    fig,ax=plt.subplots(len(ix),len(ix),figsize=(2.1*len(ix),1.7*len(ix)),squeeze=False)
    for p,j in zip(ax.flat,[y*GRID_SIDE+x for y in ix for x in ix]):
        x,z=w[j],g2[j]
        if k=='joint':p.plot(XTIME,x[:N_WAVE],label='V');p.plot(XTIME,x[N_WAVE:],label='T')
        else:p.plot(XTIME,x)
        p.set(ylim=(0,1),title=f'{z[0]:.2f},{z[1]:.2f}');p.set_xticks([])
    fig.tight_layout();fig.savefig(OUT/f'{k}_decoded_atlas.png',dpi=140);plt.show();plt.close(fig)


## 8. Observed Pareto fronts and electrical–thermal interaction
Plot original and decoded feasible points together, and nondominated fronts for each representation. Compare independent voltage-only and thermal-only exploration to joint exploration. These are finite sampled fronts, not optimized global fronts. A small crossed design independently evaluates four voltage and four thermal trajectories; compute the residual from an additive model for each completed metric only when all crossed outcomes are feasible. This tests nonseparability directly rather than inferring it from the latent projection.

In [ ]:
def nd(a):
    return np.array([not np.any(np.all(a<=v,axis=1)&np.any(a<v,axis=1)) for v in a],bool)
fig,ax=plt.subplots(2,2,figsize=(12,9))
for p,(metric,label) in zip(ax.flat,RISK.items()):
    for k in LIB:
        g=pd.concat([observed[k],decoded.get(k,pd.DataFrame())],ignore_index=True);g=g[g.feasible]
        if len(g):
            a=g[['time_min',metric]].to_numpy();p.scatter(*a.T,s=12,alpha=.3);f=a[nd(a)];f=f[np.argsort(f[:,0])];p.step(*f.T,where='post',label=k);p.scatter(*f.T,s=18)
    p.set(xlabel='Time to target (min)',ylabel=label);p.legend();p.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'pareto_comparison.png',dpi=180);plt.show()
vset=[np.full(N_WAVE,(v-V_MIN)/(V_MAX-V_MIN)) for v in [3.95,4.05,4.10,4.15]]
tset=[np.full(N_WAVE,(t-T_MIN)/(T_MAX-T_MIN)) for t in [288.15,298.15,308.15,318.15]]
crossed=[]
for i,v in enumerate(vset):
    for j,t in enumerate(tset):
        r=evaluate(protocol(np.r_[v,t],'joint'));crossed.append(dict(v_id=i,t_id=j,**r))
cross=pd.DataFrame(crossed);cross.to_csv(OUT/'crossed_design.csv',index=False)
fig,ax=plt.subplots(1,3,figsize=(13,4))
for p,metric in zip(ax,['time_min','gross_mAh','dead_mAh']):
    a=cross.pivot(index='v_id',columns='t_id',values=metric).to_numpy();feas=cross.pivot(index='v_id',columns='t_id',values='feasible').to_numpy()
    if feas.all():
        res=a-a.mean(0)[None,:]-a.mean(1)[:,None]+a.mean();print(metric,'interaction RMS',np.sqrt(np.mean(res**2)))
        q=p.imshow(res,aspect='auto',cmap='coolwarm');fig.colorbar(q,ax=p);p.set_title(metric+' additive residual')
    else:
        q=p.imshow(np.where(feas,a,np.nan),aspect='auto');fig.colorbar(q,ax=p);p.set_title(metric+' feasible only; incomplete factorial')
    p.set(xlabel='Surface temperature index',ylabel='Voltage index',xticks=range(4),yticks=range(4))
fig.tight_layout();fig.savefig(OUT/'interaction.png',dpi=160);plt.show()
import shutil
shutil.make_archive(str(OUT),'zip',OUT)
print('Download results:',str(OUT)+'.zip')


## Limitations
This notebook maps functionality; it does not yet perform DKL optimization. The VAE does not predict electrochemistry. Latent maps use sampled physical evaluations; sparse regions and numerical failures are not evidence of physical impossibility. Constant-voltage temperature-only benchmark is 4.10 V, configurable; if infeasible at the initial SOC, select a lower common voltage after examining smoke tests. Thermal boundary is discretized, with no SEI or actuator energy model. Surface temperature is an imposed environment boundary, not a resolved real surface. The library distribution and VAE beta, dimension, voltage and thermal bounds are modeling choices. The product latent projection is descriptive, not an independent held-out generalization test. Validate mass balance and state transfer, and repeat with finer thermal resolution before quantitative claims. Full runtime execution is not validated in the authoring environment.
Dense v2 uses 40×40 grids (4,800 decoded evaluations total; 5,104 total physical protocol evaluations including original and factorial, excluding smoke/calibration). A grid point is a true PyBaMM simulation, not a surrogate prediction. The four-dimensional product latent representation remains a projected scatter map, not a 40^4 grid. Persistent Drive storage is optional. Original library evaluation is not resumable in this revision; decoded grids are. A fresh retraining must reproduce the same waveform grid fingerprint to resume; if not, use a different output directory. Runtime validation remains pending.